#### Initialize

In [1]:
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
MAP_PATH = PARENT / "server/map/grid"
MAP_PATH.mkdir(parents=True, exist_ok=True)
RANKED_BUCKET_PATH = PARENT / "server/out/places_ranked"
DF_RANKED = pd.read_csv(RANKED_BUCKET_PATH / "places_scored_level_1.csv")

#### Build Region Grid

In [2]:
from server.scripts.rank_places_parse_locality.get_local import get_local_tile_id
from server.scripts.h3.h3_api import _h3_get_neighbours

df_ranked_level_2 = DF_RANKED.copy()
df_ranked_level_2['local_tile'] = df_ranked_level_2['h3_res9']#.apply(lambda row: get_local_tile_id(row), axis=1)

grid_res9 = pd.DataFrame({
    'local_tile': df_ranked_level_2['local_tile'].unique(),
}).sort_values('local_tile').reset_index(drop=True)

grid_res9['local_tiles'] = grid_res9['local_tile'].apply(
    lambda tile: _h3_get_neighbours(tile, k=2) | {tile}  # include self
)

if not (MAP_PATH / "grid_res9.csv").exists():
    for row in grid_res9.itertuples():
        places = df_ranked_level_2[df_ranked_level_2['local_tile'] == row.local_tile]
        grid_res9.at[row.Index, 'density'] = len(places)
        local_places = df_ranked_level_2[df_ranked_level_2['local_tile'].isin(row.local_tiles)]
        grid_res9.at[row.Index, 'local_density'] = len(local_places)

    grid_res9.to_csv(MAP_PATH / "grid_res9.csv", index=False)

#### Global Score

In [3]:
from server.scripts.rank_places_parse_locality.get_local import TYPE_COL, VENUE_COL, UNSPECIFIED_TYPE

# ── Global type distribution P(T | global), excluding Unspecified ───────────
group_cols = [TYPE_COL]
if VENUE_COL in df_ranked_level_2.columns:
    group_cols.append(VENUE_COL)

df_global_comp = df_ranked_level_2[df_ranked_level_2[TYPE_COL].ne(UNSPECIFIED_TYPE)]

global_composition = (
    df_global_comp.groupby(group_cols, dropna=False)
    .size()
    .rename("n_global")
    .reset_index()
)
global_composition["p_global"] = global_composition["n_global"] / max(len(df_global_comp), 1)
global_composition.drop(columns=["n_global"], inplace=True)

#### Locality Score

In [4]:
from server.scripts.rank_places_parse_locality.get_local import get_local_competition_factor
rep_ratio_per_cell = []
for _, region_row in grid_res9.iterrows():
    local_composition = get_local_competition_factor(
        df_ranked_level_2, 
        global_composition, 
        region_row['local_tiles'], 
        region_row['local_tile'],
    )
    if local_composition is not None:
        rep_ratio_per_cell.append(local_composition)
# rr - Representation Ratio
grid_df = pd.concat(rep_ratio_per_cell, ignore_index=True)
grid_df.to_csv(MAP_PATH / "grid_res9_compositions.csv", index=False)

In [5]:
# ── Join credible competition_factor back onto df_level5 ──────────────────────
join_keys = ["local_tile", TYPE_COL]
if "venueType" in df_ranked_level_2.columns and "venueType" in grid_df.columns:
    join_keys.append("venueType")

df_ranked_level_2_2 = df_ranked_level_2.merge(
    grid_df[[*join_keys, "p_local", "p_credible", "competition_factor", "representations", "neighbour_count"]],
    on=join_keys,
    how="left"
).sort_values(["local_tile"]).reset_index(drop=True)
df_ranked_level_2_2["competition_factor"] = df_ranked_level_2_2["competition_factor"].fillna(1.0)
df_ranked_level_2_2["p_credible"]         = df_ranked_level_2_2["p_credible"].fillna(0.0)
df_ranked_level_2_2["representations"]    = df_ranked_level_2_2["representations"].fillna(0).astype(int)
df_ranked_level_2_2["neighbour_count"]    = df_ranked_level_2_2["neighbour_count"].fillna(0).astype(int)
display(df_ranked_level_2_2.head(2))

,id,displayName,primaryTypeDisplayName,rating,userRatingCount,shortFormattedAddress,googleMapsUri,websiteUri,types,primaryType,...,wilson_1,normal_1,wilson_2,normal_2,local_tile,p_local,p_credible,competition_factor,representations,neighbour_count
0,ChIJnVjG4_MGdkgRPv420tn5MV8,Taste of China,Takeout Restaurant,3.4,117.0,"241 Northborough Rd, London",https://maps.google.com/?cid=68595384213920149...,https://tastesofchina.co.uk/?utm_source=GMBweb...,"['meal_takeaway', 'chinese_restaurant', 'resta...",meal_takeaway,...,0.509415,0.140620,0.481010,0.161855,89194ac3497ffff,0.250,0.030062,1.0,1,4
1,ChIJD8x54k4DdkgRh_HUL2HZjfY,Cooking Happy,Thai Restaurant,5.0,28.0,"163 Athenlay Rd, London",https://maps.google.com/?cid=17766095116484014...,https://www.cookinghappy.co.uk/,"['thai_restaurant', 'event_venue', 'restaurant...",thai_restaurant,...,0.879353,0.780859,0.808413,0.599374,89194ad000fffff,0.125,0.014848,1.0,1,8


#### Adjust Score

In [6]:
import numpy as np
from server.scripts.rank_places_parse_locality.get_local import Z_CONFIDENCE, ALPHA

# competition_boost: only amplify over-represented (competition_factor > 1), neutral otherwise.
# clip(lower=1) → no penalty for rare cuisines; they may be hidden gems.
df_ranked_level_2_2["competition_boost"] = 1 + ALPHA * np.log(df_ranked_level_2_2["competition_factor"].clip(lower=1))

for i in range(3):
    df_ranked_level_2_2[f"boosted_{i}"]    = df_ranked_level_2_2[f"wilson_{i}"] * df_ranked_level_2_2["competition_boost"]
    df_ranked_level_2_2[f"bnormal_{i}"] = df_ranked_level_2_2[f"boosted_{i}"].rank(pct=True)

print(f"Places: {len(df_ranked_level_2_2)}  |  Regions: {grid_res9.shape[0]}  |  α={ALPHA}  |  z={Z_CONFIDENCE}")
df_ranked_level_2_2.columns


Places: 13092  |  Regions: 1539  |  α=0.1  |  z=2.576


Index(['id', 'displayName', 'primaryTypeDisplayName', 'rating',
       'userRatingCount', 'shortFormattedAddress', 'googleMapsUri',
       'websiteUri', 'types', 'primaryType', 'is_chain', 'predictedType',
       'cuisineType', 'venueType', 'lat', 'lon', 'pcd', 'areacode',
       'wheelchairAccess', 'operational', 'cost', 'h3_res9', 'h3_res10',
       'wilson_0', 'normal_0', 'wilson_1', 'normal_1', 'wilson_2', 'normal_2',
       'local_tile', 'p_local', 'p_credible', 'competition_factor',
       'representations', 'neighbour_count', 'competition_boost', 'boosted_0',
       'bnormal_0', 'boosted_1', 'bnormal_1', 'boosted_2', 'bnormal_2'],
      dtype='str')

#### Export

In [7]:
df_ranked_level_2_2.drop(columns=[
    'p_credible', 'competition_boost', 'neighbour_count'
], inplace=True)

df_ranked_level_2_2.to_csv(RANKED_BUCKET_PATH / "places_scored_level_2.csv", index=False)

## SAMPLE

In [8]:
SAMPLE = df_ranked_level_2_2.copy()
SAMPLE.sort_values("wilson_1", ascending=False, inplace=True)
SAMPLE = SAMPLE[[
    "displayName", TYPE_COL, "representations", 
    "rating", "userRatingCount",
    "p_local", "wilson_1", "boosted_1", 
    "bnormal_1", "areacode"
]]
SAMPLE = SAMPLE[SAMPLE['areacode'] == 'SW12']
SAMPLE[SAMPLE['boosted_1']>SAMPLE['wilson_1']]
# SAMPLE[SAMPLE['displayName'].str.contains("milk", case=False, na=False)]

,displayName,cuisineType,representations,rating,userRatingCount,p_local,wilson_1,boosted_1,bnormal_1,areacode
1427,Wats Crackin Seafood Boil Ltd,Seafood,4,5.0,151.0,0.04878,0.975190,1.038248,0.999083,SW12
8406,Ralph’s Balham,Pizza,10,4.9,70.0,0.10989,0.906943,0.913052,0.868851,SW12
